# CineRank : Standalone MovieLens Training (Google Colab)

This notebook is deliberately **independent of the backend `app/` package**. It downloads MovieLens `ml-latest.zip`, prepares a reproducible real-user cohort, trains the recommendation models, validates the output files, and packages a backend-importable artifact bundle.

It contains no backend imports, database connection, API startup code, or server code.

## What it trains

- Popularity baseline
- TF-IDF content item vectors
- Sparse item-item kNN similarities (top-N neighbours per item; avoids a dense item × item matrix)
- Implicit ALS and BPR matrix factorization
- A target-aware sequential attention model trained with pairwise negative sampling
- Portable hybrid-ranking configuration

The bundle contains model weights, catalogue metadata, source user/movie ID mappings, the sampled real ratings, sparse matrices, version metadata, and SHA-256 checksums. The backend import script can assign permanent `title_<UUIDv7>` identifiers when it imports the catalogue.

**Important:** `TARGET_USERS=None` uses the complete MovieLens user set and can require substantial RAM and many hours for training. `TARGET_USERS=10_000` is the practical Colab starting point.

## 1. Install the training dependencies

In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.1,<3" "scipy>=1.11,<2" "scikit-learn>=1.4,<2" "torch>=2.2" "implicit" "tqdm>=4.66" "requests>=2.31" "joblib>=1.3"

## 2. Reproducible configuration

In [ ]:
from pathlib import Path
import os, sys, json, time, math, random, shutil, hashlib, zipfile, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import scipy
import sklearn
import torch
from torch import nn
import torch.nn.functional as F

try:
    from IPython.display import display
except ImportError:  # Local script execution outside notebooks.
    display = print

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# The notebook is deliberately standalone: no imports from the backend app/ package.
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-latest.zip"
WORKSPACE = Path(os.environ.get("CINERANK_WORKSPACE", "/content" if Path("/content").exists() else str(Path.cwd())))
DATA_ROOT = Path(os.environ.get("CINERANK_TRAIN_DATA_DIR", str(WORKSPACE / "cinerank_data"))).expanduser()
ML_ZIP = DATA_ROOT / "ml-latest.zip"
ML_DIR = DATA_ROOT / "ml-latest"
OUTPUT_DIR = Path(os.environ.get("CINERANK_TRAIN_OUTPUT_DIR", str(WORKSPACE / "cinerank_colab_bundle"))).expanduser()
BUNDLE_ZIP = Path(os.environ.get("CINERANK_TRAIN_BUNDLE_ZIP", str(WORKSPACE / "cinerank-colab-bundle.zip"))).expanduser()

def _optional_int(value):
    value = str(value).strip().lower()
    return None if value in {"", "none", "null", "all"} else int(value)

TARGET_USERS = _optional_int(os.environ.get("CINERANK_TARGET_USERS", "10000"))
MIN_MOVIE_RATINGS = int(os.environ.get("CINERANK_MIN_MOVIE_RATINGS", "20"))
if TARGET_USERS is not None and TARGET_USERS < 1:
    raise ValueError("CINERANK_TARGET_USERS must be a positive integer or 'none'.")
if MIN_MOVIE_RATINGS < 1:
    raise ValueError("CINERANK_MIN_MOVIE_RATINGS must be >= 1.")

RATINGS_CHUNK_SIZE = 1_000_000
ITEMKNN_NEIGHBORS = 50
ALS_FACTORS, ALS_ITERATIONS, ALS_REG = 64, 15, 0.05
BPR_FACTORS, BPR_ITERATIONS, BPR_LEARNING_RATE = 64, 40, 0.001
ATTENTION_EMBED_DIM = 128
ATTENTION_MAX_HISTORY = 100
ATTENTION_EPOCHS = 12
ATTENTION_BATCH_SIZE = 512
ATTENTION_LEARNING_RATE = 1e-3
ATTENTION_MAX_TRAIN_PAIRS = 300_000
ATTENTION_SCORE_CHUNK_SIZE = 4096

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
BUNDLE_ZIP.parent.mkdir(parents=True, exist_ok=True)
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        f"Output directory is not empty: {OUTPUT_DIR}. Choose a fresh run directory; "
        "the notebook will not delete previous training artifacts."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({"device": str(DEVICE), "target_users": TARGET_USERS,
       "min_movie_ratings": MIN_MOVIE_RATINGS,
       "output_dir": str(OUTPUT_DIR), "data_dir": str(DATA_ROOT)})


## 3. Download and safely extract `ml-latest.zip`

In [ ]:
def download_file(url: str, destination: Path, chunk_size: int = 1 << 20,
                  connect_timeout: int = 30, read_timeout: int = 300) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Using existing download: {destination} ({destination.stat().st_size:,} bytes)")
        return destination

    tmp_path = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=(connect_timeout, read_timeout)) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with open(tmp_path, "wb") as handle, tqdm(
            total=total or None, unit="B", unit_scale=True, desc="MovieLens download"
        ) as progress:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    handle.write(chunk)
                    progress.update(len(chunk))
    tmp_path.replace(destination)
    return destination


def safe_extract_zip(archive: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f"Unsafe ZIP member path: {member.filename}")
        zf.extractall(destination)

if not (ML_DIR / "ratings.csv").exists() or not (ML_DIR / "movies.csv").exists():
    if ML_ZIP.exists() and not zipfile.is_zipfile(ML_ZIP):
        ML_ZIP.unlink()
    download_file(MOVIELENS_URL, ML_ZIP)
    if not zipfile.is_zipfile(ML_ZIP):
        raise RuntimeError("Downloaded MovieLens archive is not a valid ZIP file")
    safe_extract_zip(ML_ZIP, DATA_ROOT)

assert (ML_DIR / "ratings.csv").is_file(), "ratings.csv missing after extraction"
assert (ML_DIR / "movies.csv").is_file(), "movies.csv missing after extraction"
print("MovieLens files ready:", ML_DIR)

## 4. Sample real users and filter long-tail movies

In [ ]:
def read_movielens_subset() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Sample real users, filter long-tail items, then create a temporal holdout.

    The final rating per user is held out only for users with at least three
    retained ratings. All model fitting and imported training history excludes
    those holdout rows, so offline top-K evaluation is not contaminated.
    """
    ratings_path = ML_DIR / "ratings.csv"
    movies_path = ML_DIR / "movies.csv"

    # First pass gets the true user population without loading 33M rows at once.
    user_ids = set()
    total_rows = 0
    for chunk in pd.read_csv(ratings_path, usecols=["userId"],
                             dtype={"userId": "int32"}, chunksize=RATINGS_CHUNK_SIZE):
        user_ids.update(chunk["userId"].unique().tolist())
        total_rows += len(chunk)
    all_users = np.asarray(sorted(user_ids), dtype=np.int64)
    print(f"Raw dataset: {total_rows:,} ratings, {len(all_users):,} real users")

    if TARGET_USERS is None or TARGET_USERS >= len(all_users):
        selected_users = set(map(int, all_users))
        print("Using all MovieLens users (full-data mode).")
    else:
        rng = np.random.default_rng(SEED)
        selected_users = set(map(int, rng.choice(all_users, size=TARGET_USERS, replace=False)))
        print(f"Sampled {len(selected_users):,} real users; no synthetic users are added.")

    # Second pass retains only the sampled real users.
    pieces = []
    columns = ["userId", "movieId", "rating", "timestamp"]
    dtypes = {"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"}
    for chunk in pd.read_csv(ratings_path, usecols=columns, dtype=dtypes,
                             chunksize=RATINGS_CHUNK_SIZE):
        part = chunk[chunk["userId"].isin(selected_users)]
        if not part.empty:
            pieces.append(part)
    if not pieces:
        raise RuntimeError("No ratings remained after user sampling.")
    ratings = pd.concat(pieces, ignore_index=True)
    del pieces

    # Apply the requested long-tail threshold to the sampled, real rating set.
    movie_counts = ratings.groupby("movieId", sort=False).size()
    eligible_movie_ids = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index
    ratings = ratings[ratings["movieId"].isin(eligible_movie_ids)].copy()

    movies = pd.read_csv(movies_path, dtype={"movieId": "int32"})
    movies = movies[movies["movieId"].isin(eligible_movie_ids)].copy()
    movies["year"] = pd.to_numeric(
        movies["title"].str.extract(r"\((\d{4})\)", expand=False), errors="coerce"
    ).astype("Int64")

    # Exact requested relevance mapping, applied uniformly to all real ratings.
    ratings["relevance"] = ((ratings["rating"] - 0.5) / 4.5).clip(0.0, 1.0).astype("float32")
    ratings.sort_values(["userId", "movieId", "timestamp"], inplace=True, kind="stable")
    ratings.drop_duplicates(["userId", "movieId"], keep="last", inplace=True)

    # Deterministic temporal leave-one-out holdout (one relevant target per eligible user).
    ratings.sort_values(["userId", "timestamp", "movieId"], inplace=True, kind="stable")
    user_counts = ratings.groupby("userId", sort=False).size()
    eligible_users = set(user_counts[user_counts >= 3].index.astype(int))
    last_rows = ratings.groupby("userId", sort=False).tail(1)
    holdout = last_rows[last_rows["userId"].isin(eligible_users)].copy()
    train = ratings.drop(index=holdout.index).copy()

    # A held-out user necessarily retains >=2 training events.
    retained_users = np.sort(train["userId"].unique())
    train = train[train["userId"].isin(retained_users)].copy()
    movies.sort_values("movieId", inplace=True, kind="stable")
    movies.reset_index(drop=True, inplace=True)
    train.sort_values(["userId", "timestamp", "movieId"], inplace=True, kind="stable")
    holdout.sort_values(["userId", "timestamp", "movieId"], inplace=True, kind="stable")
    train.reset_index(drop=True, inplace=True)
    holdout.reset_index(drop=True, inplace=True)

    if movies.empty or train.empty:
        raise RuntimeError("Filtering removed all movies or training ratings. Lower MIN_MOVIE_RATINGS.")
    print(f"Training cohort: {len(train):,} ratings, {train.userId.nunique():,} users, "
          f"{len(movies):,} movies; held out {len(holdout):,} latest ratings.")
    return train, movies, holdout

ratings_df, movies_df, holdout_df = read_movielens_subset()
display(movies_df.head())
display(ratings_df.head())
display(holdout_df.head())


## 5. Build dense index maps and sparse matrices

In [ ]:
source_user_ids = np.sort(ratings_df["userId"].unique().astype(np.int64))
source_movie_ids = movies_df["movieId"].to_numpy(dtype=np.int64)
user_to_idx = {int(uid): i for i, uid in enumerate(source_user_ids)}
movie_to_idx = {int(mid): i for i, mid in enumerate(source_movie_ids)}

ratings_df["user_idx"] = ratings_df["userId"].map(user_to_idx).astype("int32")
ratings_df["item_idx"] = ratings_df["movieId"].map(movie_to_idx).astype("int32")

# Confidence matrix: one positive observation per real rating, weighted by relevance.
# `relevance` itself is also preserved exactly in relevance_mat.npz and ratings_sample.csv.
confidence = (1.0 + 4.0 * ratings_df["relevance"].to_numpy(dtype=np.float32))
rows = ratings_df["user_idx"].to_numpy(dtype=np.int32)
cols = ratings_df["item_idx"].to_numpy(dtype=np.int32)
shape = (len(source_user_ids), len(source_movie_ids))
train_mat = csr_matrix((confidence, (rows, cols)), shape=shape, dtype=np.float32)
relevance_mat = csr_matrix((ratings_df["relevance"].to_numpy(dtype=np.float32),
                            (rows, cols)), shape=shape, dtype=np.float32)
train_mat.sort_indices()
relevance_mat.sort_indices()

assert train_mat.shape == shape
assert train_mat.nnz == len(ratings_df)
print("Training matrix:", train_mat.shape, "observations:", train_mat.nnz)
print("Matrix memory (MB):", round((train_mat.data.nbytes + train_mat.indices.nbytes + train_mat.indptr.nbytes) / 1e6, 2))

## 6. Train popularity, content TF-IDF, and sparse ItemKNN

In [ ]:
# Popularity baseline. Store the source signal and its stable item-index ordering.
popularity_scores = np.asarray(train_mat.sum(axis=0)).ravel().astype(np.float32)
np.savez_compressed(OUTPUT_DIR / "rec-pop-v1.npz", scores=popularity_scores)

# The backend uses these fixed per-item vectors to form a history profile.
movie_texts = (
    movies_df["title"].fillna("").astype(str) + " " +
    movies_df["genres"].fillna("").astype(str).str.replace("|", " ", regex=False)
).tolist()
vectorizer = TfidfVectorizer(max_features=50_000, stop_words="english", ngram_range=(1, 2),
                             min_df=1, dtype=np.float32)
content_vectors = vectorizer.fit_transform(movie_texts).tocsr().astype(np.float32)
save_npz(OUTPUT_DIR / "rec-content-v1.npz", content_vectors, compressed=True)
content_meta = {
    "schema_version": 1,
    "vocabulary": {str(token): int(index) for token, index in vectorizer.vocabulary_.items()},
    "idf": vectorizer.idf_.astype(np.float32).tolist(),
    "ngram_range": [1, 2],
    "stop_words": "english",
    "norm": "l2",
}
(OUTPUT_DIR / "rec-content-v1-meta.json").write_text(
    json.dumps(content_meta, ensure_ascii=False, separators=(",", ":")), encoding="utf-8"
)


# Sparse item-item cosine similarities in bounded row blocks.
# No dense n_items x n_items matrix is materialised in Python memory.
def build_topk_itemknn(user_item: csr_matrix, k: int, row_block: int = 64) -> csr_matrix:
    item_user = normalize(user_item.T.tocsr(), norm="l2", axis=1, copy=True)
    n_items = item_user.shape[0]
    out_rows, out_cols, out_data = [], [], []
    for block_start in tqdm(range(0, n_items, row_block), desc="ItemKNN row blocks"):
        block_end = min(n_items, block_start + row_block)
        similarities = (item_user[block_start:block_end] @ item_user.T).tocsr()
        for local_row in range(block_end - block_start):
            global_row = block_start + local_row
            start, end = similarities.indptr[local_row:local_row + 2]
            col_idx = similarities.indices[start:end]
            values = similarities.data[start:end]
            keep_mask = col_idx != global_row
            col_idx, values = col_idx[keep_mask], values[keep_mask]
            if values.size == 0:
                continue
            keep_n = min(k, values.size)
            chosen = np.argpartition(values, -keep_n)[-keep_n:]
            chosen = chosen[np.argsort(values[chosen])[::-1]]
            out_rows.extend([global_row] * len(chosen))
            out_cols.extend(col_idx[chosen].tolist())
            out_data.extend(values[chosen].astype(np.float32).tolist())
        del similarities
    result = csr_matrix((np.asarray(out_data, dtype=np.float32),
                         (np.asarray(out_rows, dtype=np.int32), np.asarray(out_cols, dtype=np.int32))),
                        shape=(n_items, n_items), dtype=np.float32)
    result.sort_indices()
    return result

itemknn_similarity = build_topk_itemknn(train_mat, ITEMKNN_NEIGHBORS, row_block=64)
save_npz(OUTPUT_DIR / "rec-itemknn-v1.npz", itemknn_similarity, compressed=True)
print("Content matrix:", content_vectors.shape, "ItemKNN nnz:", itemknn_similarity.nnz)

## 7. Train implicit ALS and BPR

In [ ]:
!pip install implicit

In [ ]:
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

# implicit expects a user × item matrix. Do NOT transpose this matrix.
als_model = AlternatingLeastSquares(
    factors=ALS_FACTORS, regularization=ALS_REG, iterations=ALS_ITERATIONS,
    random_state=SEED, use_gpu=False
)
als_model.fit(train_mat, show_progress=True)
np.savez_compressed(OUTPUT_DIR / "rec-als-v1.npz",
                    user_factors=als_model.user_factors.astype(np.float32),
                    item_factors=als_model.item_factors.astype(np.float32),
                    factors=np.asarray([ALS_FACTORS], dtype=np.int32))

del als_model

bpr_model = BayesianPersonalizedRanking(
    factors=BPR_FACTORS, learning_rate=BPR_LEARNING_RATE,
    iterations=BPR_ITERATIONS, random_state=SEED, use_gpu=False
)
bpr_model.fit(train_mat, show_progress=True)
np.savez_compressed(OUTPUT_DIR / "rec-bpr-v1.npz",
                    user_factors=bpr_model.user_factors.astype(np.float32),
                    item_factors=bpr_model.item_factors.astype(np.float32),
                    factors=np.asarray([BPR_FACTORS], dtype=np.int32))
del bpr_model
print("Saved ALS and BPR factor matrices.")

## 8. Train target-aware sequential attention

In [ ]:
class TargetAwareAttention(nn.Module):
    """Small DIN-like scorer. Item index 0 is reserved for history padding."""
    def __init__(self, n_items: int, emb_dim: int = 64, max_history: int = 30):
        super().__init__()
        self.item_embedding = nn.Embedding(n_items + 1, emb_dim, padding_idx=0)
        self.attention = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim), nn.ReLU(), nn.Linear(emb_dim, 1)
        )
        self.scorer = nn.Sequential(
            nn.Linear(emb_dim * 3, emb_dim), nn.ReLU(), nn.Linear(emb_dim, 1)
        )
        self.max_history = max_history
        self.emb_dim = emb_dim

    def forward(self, histories: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        h = self.item_embedding(histories)             # [B, L, D]
        t = self.item_embedding(targets).unsqueeze(1)  # [B, 1, D]
        attn_logits = self.attention(torch.cat([h, t.expand_as(h)], dim=-1)).squeeze(-1)
        mask = histories.ne(0)
        attn_logits = attn_logits.masked_fill(~mask, -1e9)
        weights = torch.softmax(attn_logits, dim=1)
        weights = torch.where(mask.any(dim=1, keepdim=True), weights, torch.zeros_like(weights))
        context = torch.sum(weights.unsqueeze(-1) * h, dim=1)
        target = t.squeeze(1)
        features = torch.cat([context, target, context * target], dim=-1)
        return self.scorer(features).squeeze(-1)


def make_attention_examples(frame: pd.DataFrame, max_history: int, max_pairs: int, seed: int):
    rng = np.random.default_rng(seed)
    grouped = frame.sort_values(["user_idx", "timestamp", "item_idx"], kind="stable").groupby("user_idx")
    eligible = []
    seen_by_user = {}
    for user_idx, group in grouped:
        seq = group["item_idx"].to_numpy(dtype=np.int32)
        if len(seq) >= 2:
            eligible.append((int(user_idx), seq))
            seen_by_user[int(user_idx)] = set(map(int, seq))
    if not eligible:
        raise RuntimeError("Not enough per-user sequence history for attention training.")

    offsets = np.cumsum([0] + [len(seq) - 1 for _, seq in eligible])
    total_pairs = int(offsets[-1])
    n_examples = min(total_pairs, max_pairs)
    flat_positions = (np.arange(total_pairs, dtype=np.int64) if total_pairs <= max_pairs
                      else np.sort(rng.choice(total_pairs, size=max_pairs, replace=False)))
    group_positions = np.searchsorted(offsets, flat_positions, side="right") - 1

    n_items = len(source_movie_ids)
    histories = np.zeros((n_examples, max_history), dtype=np.int32)
    positives = np.zeros(n_examples, dtype=np.int64)
    negatives = np.zeros(n_examples, dtype=np.int64)
    kept = 0
    for flat, group_idx in tqdm(zip(flat_positions, group_positions), total=n_examples,
                                desc="Building attention examples"):
        user_idx, seq = eligible[int(group_idx)]
        pos = int(flat - offsets[group_idx] + 1)
        hist = seq[max(0, pos - max_history):pos]
        if len(hist):
            histories[kept, -len(hist):] = hist + 1  # 0 is padding
        positive = int(seq[pos]) + 1
        seen = seen_by_user[user_idx]
        negative = None
        for _ in range(100):
            candidate = int(rng.integers(0, n_items))
            if candidate not in seen:
                negative = candidate + 1
                break
        if negative is None:
            # Extremely dense user: find a valid negative once, then skip if none.
            for candidate in range(n_items):
                if candidate not in seen:
                    negative = candidate + 1
                    break
        if negative is None:
            continue
        positives[kept] = positive
        negatives[kept] = negative
        kept += 1

    if kept == 0:
        raise RuntimeError("No valid negative examples for attention training.")
    return histories[:kept], positives[:kept], negatives[:kept]

hist_array, pos_array, neg_array = make_attention_examples(
    ratings_df, ATTENTION_MAX_HISTORY, ATTENTION_MAX_TRAIN_PAIRS, SEED
)
attention_model = TargetAwareAttention(len(source_movie_ids), ATTENTION_EMBED_DIM,
                                       ATTENTION_MAX_HISTORY).to(DEVICE)
optimizer = torch.optim.AdamW(attention_model.parameters(), lr=ATTENTION_LEARNING_RATE,
                              weight_decay=1e-5)

for epoch in range(ATTENTION_EPOCHS):
    permutation = np.random.permutation(len(pos_array))
    attention_model.train()
    losses = []
    for start in tqdm(range(0, len(permutation), ATTENTION_BATCH_SIZE),
                      desc=f"Attention epoch {epoch+1}/{ATTENTION_EPOCHS}"):
        idx = permutation[start:start + ATTENTION_BATCH_SIZE]
        h = torch.as_tensor(hist_array[idx], dtype=torch.long, device=DEVICE)
        p = torch.as_tensor(pos_array[idx], dtype=torch.long, device=DEVICE)
        n = torch.as_tensor(neg_array[idx], dtype=torch.long, device=DEVICE)
        pos_score = attention_model(h, p)
        neg_score = attention_model(h, n)
        loss = -F.logsigmoid(pos_score - neg_score).mean()
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(attention_model.parameters(), max_norm=5.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    print(f"Epoch {epoch+1}: pairwise_loss={np.mean(losses):.5f}")

attention_model = attention_model.to("cpu").eval()
torch.save({
    "state_dict": {k: v.detach().cpu() for k, v in attention_model.state_dict().items()},
    "n_items": int(len(source_movie_ids)),
    "emb_dim": ATTENTION_EMBED_DIM,
    "max_history": ATTENTION_MAX_HISTORY,
    "architecture": "target_aware_attention_v1",
}, OUTPUT_DIR / "rec-attn-v1.pt")
print("Saved portable attention state dictionary.")

## 9. Save matrices, maps, catalogue, sampled ratings, and hybrid settings

In [ ]:
save_npz(OUTPUT_DIR / "train-mat-v1.npz", train_mat, compressed=True)
save_npz(OUTPUT_DIR / "relevance-mat-v1.npz", relevance_mat, compressed=True)

id_maps = {
    "schema_version": 2,
    "matrix_orientation": "users_by_items",
    "idx_to_source_user_id": [int(x) for x in source_user_ids],
    "idx_to_movielens_movie_id": [int(x) for x in source_movie_ids],
    "source_user_id_to_idx": {str(int(uid)): int(i) for i, uid in enumerate(source_user_ids)},
    "movielens_movie_id_to_idx": {str(int(mid)): int(i) for i, mid in enumerate(source_movie_ids)},
}
with open(OUTPUT_DIR / "id-maps-v1.json", "w", encoding="utf-8") as handle:
    json.dump(id_maps, handle, ensure_ascii=False, separators=(",", ":"))

catalogue_export = movies_df[["movieId", "title", "genres", "year"]].copy()
catalogue_export.rename(columns={"movieId": "movielens_id"}, inplace=True)
catalogue_export["media_type"] = "movie"
catalogue_export["year"] = catalogue_export["year"].astype("Int64")
catalogue_export.to_csv(OUTPUT_DIR / "catalogue.csv", index=False)

ratings_export = ratings_df[["userId", "movieId", "rating", "relevance", "timestamp"]].copy()
ratings_export.rename(columns={"userId": "source_user_id", "movieId": "movielens_id"}, inplace=True)
ratings_export.to_csv(OUTPUT_DIR / "ratings_sample.csv", index=False)

holdout_export = holdout_df[["userId", "movieId", "rating", "relevance", "timestamp"]].copy()
holdout_export.rename(columns={"userId": "source_user_id", "movieId": "movielens_id"}, inplace=True)
holdout_export.to_csv(OUTPUT_DIR / "holdout_sample.csv", index=False)

hybrid_config = {
    "schema_version": 1,
    "algorithm": "weighted_score_fusion_with_mmr",
    "weights": {"attention": 0.30, "als": 0.25, "bpr": 0.15,
                "itemknn": 0.10, "content": 0.10, "popularity": 0.10},
    "diversity_lambda": 0.15,
    "score_normalization": "per-source percentile/rank normalization at serving time",
}
with open(OUTPUT_DIR / "rec-hybrid-v1.json", "w", encoding="utf-8") as handle:
    json.dump(hybrid_config, handle, indent=2)

print("Exported files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size/1e6:8.2f} MB")

## 10. Validate the artifact bundle and write checksums

In [ ]:
# Structural checks catch item-index mismatches before the bundle reaches the backend.
assert train_mat.shape == (len(source_user_ids), len(source_movie_ids))
assert relevance_mat.shape == train_mat.shape
assert content_vectors.shape[0] == len(source_movie_ids)
assert itemknn_similarity.shape == (len(source_movie_ids), len(source_movie_ids))
assert popularity_scores.shape[0] == len(source_movie_ids)

als_check = np.load(OUTPUT_DIR / "rec-als-v1.npz")
bpr_check = np.load(OUTPUT_DIR / "rec-bpr-v1.npz")
assert als_check["user_factors"].shape[0] == len(source_user_ids)
assert als_check["item_factors"].shape[0] == len(source_movie_ids)
assert bpr_check["user_factors"].shape[0] == len(source_user_ids)
assert bpr_check["item_factors"].shape[0] == len(source_movie_ids)

metadata = {
    "bundle_format": "cinerank-portable-v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "MovieLens ml-latest.zip",
    "dataset_url": MOVIELENS_URL,
    "seed": SEED,
    "target_users": TARGET_USERS,
    "min_movie_ratings": MIN_MOVIE_RATINGS,
    "n_users": int(len(source_user_ids)),
    "n_items": int(len(source_movie_ids)),
    "n_ratings": int(len(ratings_df)),
    "n_holdout_ratings": int(len(holdout_df)),
    "holdout_policy": "latest timestamp rating per user with >=3 retained ratings",
    "matrix_shape": list(train_mat.shape),
    "matrix_nnz": int(train_mat.nnz),
    "relevance_mapping": "clip((rating - 0.5) / 4.5, 0, 1)",
    "matrix_values": "confidence = 1 + 4 * relevance; all observed ratings remain positive",
    "models": ["popularity", "content_tfidf", "itemknn_topk", "implicit_als", "implicit_bpr", "target_aware_attention", "hybrid_config"],
    "hyperparameters": {
        "itemknn_neighbors": ITEMKNN_NEIGHBORS,
        "als_factors": ALS_FACTORS, "als_iterations": ALS_ITERATIONS,
        "bpr_factors": BPR_FACTORS, "bpr_iterations": BPR_ITERATIONS,
        "attention_emb_dim": ATTENTION_EMBED_DIM,
        "attention_max_history": ATTENTION_MAX_HISTORY,
        "attention_epochs": ATTENTION_EPOCHS,
        "attention_train_pairs": int(len(pos_array)),
    },
    "environment": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__, "torch": torch.__version__,
    },
}
# Stable identity for the uncompressed model files; checksum manifest is computed next.
bundle_parts = []
for artifact in sorted(OUTPUT_DIR.iterdir()):
    if artifact.is_file() and artifact.name not in {"manifest.json", "SHA256SUMS"}:
        bundle_parts.append(artifact.name.encode() + b"\0" + hashlib.sha256(artifact.read_bytes()).digest())
metadata["bundle_id"] = hashlib.sha256(b"".join(bundle_parts)).hexdigest()[:32]
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

checksums = {}
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file() and path.name != "SHA256SUMS":
        digest = hashlib.sha256()
        with open(path, "rb") as stream:
            for chunk in iter(lambda: stream.read(1 << 20), b""):
                digest.update(chunk)
        checksums[path.name] = digest.hexdigest()
with open(OUTPUT_DIR / "SHA256SUMS", "w", encoding="utf-8") as handle:
    for filename, digest in checksums.items():
        handle.write(f"{digest}  {filename}\n")

print("Bundle validated. SHA-256 files:", len(checksums))

## 11. Package bundle and download from Colab

In [ ]:
if BUNDLE_ZIP.exists():
    BUNDLE_ZIP.unlink()
with zipfile.ZipFile(BUNDLE_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            zf.write(path, arcname=f"cinerank_colab_bundle/{path.name}")

print(f"Created {BUNDLE_ZIP} ({BUNDLE_ZIP.stat().st_size/1e6:.2f} MB)")

# This cell runs only in Google Colab; outside Colab, use the file path above.
try:
    from google.colab import files
    files.download(str(BUNDLE_ZIP))
except ImportError:
    print("Not running in Google Colab. Bundle is available at:", BUNDLE_ZIP)

## 12. Optional: persist a copy to Google Drive

Run this cell only when you want a second copy in Drive. The browser download above is independent.

In [ ]:
SAVE_TO_GOOGLE_DRIVE = False
GOOGLE_DRIVE_DESTINATION = "/content/drive/MyDrive/CineRank/cinerank-colab-bundle.zip"

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path(GOOGLE_DRIVE_DESTINATION)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE_ZIP, destination)
    print("Copied bundle to:", destination)
else:
    print("Drive copy disabled. Set SAVE_TO_GOOGLE_DRIVE=True to enable it.")

## Colab tuning notes

- If training is too slow, set `TARGET_USERS=5_000` or increase `MIN_MOVIE_RATINGS=50`.
- If using a high-memory GPU/runtime and you need a larger model, set `TARGET_USERS=None` and consider `MIN_MOVIE_RATINGS=5`. Full ALS/BPR and the attention model may take hours and require substantial RAM.
- `ITEMKNN_NEIGHBORS` caps stored item neighbours. Increasing it improves candidate recall at the cost of storage and inference work.
- `ATTENTION_MAX_TRAIN_PAIRS` controls attention examples independently of the matrix size. Increase it when you have sufficient GPU memory/time.
- Do not treat MovieLens ratings as clicks or watch events without qualification. This notebook trains preference models from explicit ratings; implicit ALS/BPR consume the observed interactions as positive events weighted by mapped relevance.

## Backend import

After downloading the ZIP, extract it into the backend project and run:

```bash
python scripts/import_colab_bundle.py --bundle /path/to/cinerank-colab-bundle.zip
```

The backend importer creates immutable `title_<UUIDv7>` identifiers for catalogue rows and maps MovieLens source IDs separately. Re-running the import is designed to upsert the same source rows rather than assign new title identifiers. Start the API only after the import has completed.